In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# --- Educational Note on Dynamic Interactive Transformations ---
display(Markdown(r"""
> **Educational Note:**
> Αυτό το notebook υποστηρίζει **αλυσιδωτούς μετασχηματισμούς** με δυναμικό έλεγχο των widgets:
> * **Πάνω Γράφημα:** Το αρχικό σήμα αναφοράς $x[n] = (6 - n)(u[n] - u[n - 6])$.
> * **Κάτω Γράφημα:** Το τρέχον μετασχηματισμένο σήμα. 
> * **Dynamic UI:** Τα sliders ενεργοποιούνται ή απενεργοποιούνται αυτόματα ανάλογα με τον μετασχηματισμό που επιλέγεις από τη λίστα.
"""))

# Ορισμός του αρχικού σήματος x[n] = (6 - n) * (u[n] - u[n - 6])
def compute_x_base(n_arr):
    u = (n_arr >= 0).astype(float) - (n_arr >= 6).astype(float)
    return (6 - n_arr) * u

# Κλάση διαχείρισης της κατάστασης (State) της αλυσίδας μετασχηματισμών
class SignalPipeline:
    def __init__(self):
        self.reset()
        
    def reset(self):
        self.n_base = np.arange(-40, 41)
        self.current_values = compute_x_base(self.n_base)
        self.history_desc = ["x[n]"]
        
    def apply_transform(self, t_type, k, a):
        n = self.n_base
        prev_vals = self.current_values
        new_vals = np.zeros_like(n, dtype=float)
        
        if t_type == 'shift':
            for i, val in enumerate(n):
                target = val - k
                match = np.where(n == target)[0]
                if len(match) > 0:
                    new_vals[i] = prev_vals[match[0]]
            sign_str = f"- {k}" if k >= 0 else f"+ {abs(k)}"
            new_desc = f"Shift({self.history_desc[-1]}, n {sign_str})"
            
        elif t_type == 'reverse':
            for i, val in enumerate(n):
                target = -val
                match = np.where(n == target)[0]
                if len(match) > 0:
                    new_vals[i] = prev_vals[match[0]]
            new_desc = f"Reverse({self.history_desc[-1]})"
            
        elif t_type == 'scale':
            for i, val in enumerate(n):
                target = a * val
                match = np.where(n == target)[0]
                if len(match) > 0:
                    new_vals[i] = prev_vals[match[0]]
            new_desc = f"Scale({self.history_desc[-1]}, {a}n)"
            
        self.current_values = new_vals
        self.history_desc.append(new_desc)

pipeline = SignalPipeline()

# Δημιουργία Widgets
transform_dropdown = widgets.Dropdown(
    options=[
        ('Time Shifting: [n - k]', 'shift'),
        ('Time Reversal: [-n]', 'reverse'),
        ('Time Scaling: [a * n]', 'scale')
    ],
    value='shift',
    description='Next Transform:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='550px')
)

k_slider = widgets.IntSlider(
    value=2, min=-10, max=10, step=1, 
    description='Shift (k):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='550px')
)

alpha_slider = widgets.IntSlider(
    value=2, min=1, max=4, step=1, 
    description='Scale (a):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='550px')
)

n_range_slider = widgets.IntSlider(
    value=12, min=6, max=25, step=1, 
    description=r'View Range ($\pm n$):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='550px')
)

apply_button = widgets.Button(
    description='Apply to Current Signal', 
    button_style='success', 
    layout=widgets.Layout(width='260px')
)

reset_button = widgets.Button(
    description='Reset Pipeline', 
    button_style='danger', 
    layout=widgets.Layout(width='260px')
)

output_widget = widgets.Output()

# Συνάρτηση που ενημερώνει ποια sliders είναι ενεργά/ανενεργά ανάλογα με την επιλογή
def on_transform_change(change):
    selected = change['new']
    if selected == 'shift':
        k_slider.disabled = False
        alpha_slider.disabled = True
    elif selected == 'reverse':
        k_slider.disabled = True
        alpha_slider.disabled = True
    elif selected == 'scale':
        k_slider.disabled = True
        alpha_slider.disabled = False

transform_dropdown.observe(on_transform_change, names='value')

# Αρχική κατάσταση widgets (καθώς ξεκινάμε με 'shift')
alpha_slider.disabled = True

# Συνάρτηση σχεδίασης και ενημέρωσης
def update_plot():
    with output_widget:
        output_widget.clear_output(wait=True)
        
        n_rng = n_range_slider.value
        n = pipeline.n_base
        mask = (n >= -n_rng) & (n <= n_rng)
        
        n_plot = n[mask]
        orig_plot = compute_x_base(n_plot)
        curr_plot = pipeline.current_values[mask]
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
        
        # --- Top Plot: Original Reference Signal ---
        ax1.stem(n_plot, orig_plot, linefmt='b-', markerfmt='bo', basefmt='k-')
        ax1.set_title(r'Original Reference Signal: $x[n] = (6 - n)(u[n] - u[n - 6])$', fontsize=11, fontweight='bold', color='darkblue')
        ax1.set_ylabel('Amplitude', fontsize=10)
        ax1.grid(True, linestyle='--', alpha=0.6)
        ax1.set_ylim(-1.5, 7.5)
        
        # --- Bottom Plot: Cascaded Transformed Signal ---
        current_title = pipeline.history_desc[-1]
        ax2.stem(n_plot, curr_plot, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax2.set_title(f'Cascaded Result: {current_title}', fontsize=11, fontweight='bold', color='darkred')
        ax2.set_xlabel('Index $n$', fontsize=10)
        ax2.set_ylabel('Amplitude', fontsize=10)
        ax2.grid(True, linestyle='--', alpha=0.6)
        ax2.set_ylim(-1.5, 7.5)
        
        plt.tight_layout()
        plt.show()

# Callbacks για τα κουμπιά
def on_apply_clicked(b):
    pipeline.apply_transform(transform_dropdown.value, k_slider.value, alpha_slider.value)
    update_plot()

def on_reset_clicked(b):
    pipeline.reset()
    update_plot()

apply_button.on_click(on_apply_clicked)
reset_button.on_click(on_reset_clicked)

n_range_slider.observe(lambda change: update_plot(), names='value')

# Αρχική εκτέλεση σχεδίασης
update_plot()

# Διάταξη UI
display(widgets.VBox([
    transform_dropdown,
    widgets.HBox([k_slider, alpha_slider]),
    n_range_slider,
    widgets.HBox([apply_button, reset_button]),
    output_widget
]))


> **Educational Note:**
> Αυτό το notebook υποστηρίζει **αλυσιδωτούς μετασχηματισμούς** με δυναμικό έλεγχο των widgets:
> * **Πάνω Γράφημα:** Το αρχικό σήμα αναφοράς $x[n] = (6 - n)(u[n] - u[n - 6])$.
> * **Κάτω Γράφημα:** Το τρέχον μετασχηματισμένο σήμα. 
> * **Dynamic UI:** Τα sliders ενεργοποιούνται ή απενεργοποιούνται αυτόματα ανάλογα με τον μετασχηματισμό που επιλέγεις από τη λίστα.
